# 02 - Data Cleaning

This notebook cleans and prepares the raw data for merging and feature engineering.

**Input:** Raw data from `data/raw/elap733/` and `data/raw/nba_api/`

**Output:** Cleaned datasets in `data/processed/`:
- `injury_history_by_player_season.csv` — Aggregated injury data (TARGET VARIABLE)
- `player_stats_combined.csv` — Combined player stats across all seasons
- `player_bio_combined.csv` — Combined player bio data
- `tracking_stats_combined.csv` — Player tracking data (2013+ only)
- `team_back_to_backs.csv` — Back-to-back game counts per team-season
- `team_schedules_with_b2b.csv` — Game-level schedule with B2B flags
- `player_id_mapping.csv` — Mapping between injury-data names and NBA player IDs

In [1]:
import sys
sys.path.append('..')

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# For fuzzy matching
from difflib import SequenceMatcher

from src.config import (
    FIRST_SEASON, LAST_SEASON,
    RAW_ELAP_DIR, RAW_NBA_API_DIR,
    PROCESSED_DIR, TRACKING_DATA_START
)

from src.utils import (
    MANUAL_NAME_MAPPINGS,
    EXCLUDE_FROM_INJURY_DATA
)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Ensure output directory exists
Path(f'../{PROCESSED_DIR}').mkdir(parents=True, exist_ok=True)

print(f"Season range: {FIRST_SEASON} to {LAST_SEASON}")
print(f"Output directory: {PROCESSED_DIR}")
print(f"Manual name mappings: {len(MANUAL_NAME_MAPPINGS)}")
print(f"Excluded names (coaches/staff): {len(EXCLUDE_FROM_INJURY_DATA)}")

Season range: 2010 to 2019
Output directory: data/processed
Manual name mappings: 48
Excluded names (coaches/staff): 16


---
## Section 1: Load All Raw Data

We load only the raw files that this pipeline actually uses:
- **elap733** `missed_games` — the source for our injury target variable
- **nba_api** player stats, player bio, tracking stats, and team schedules

Files like `injury_transactions` and elap733's own player stats / schedules are **not loaded** because they duplicate nba_api data or are unused downstream.

In [2]:
# Load the one elap733 file we need: missed games (our injury target source)
print("Loading elap733 missed games data...")
print("=" * 60)

df_missed_games_raw = pd.read_csv(f'../{RAW_ELAP_DIR}/missed_games_2010_2019.csv', index_col=0)
print(f"missed_games: {df_missed_games_raw.shape}")

Loading elap733 missed games data...
missed_games: (11531, 5)


In [3]:
# Survey all raw nba_api files (quick check before processing)
print("Raw nba_api files available:")
print("=" * 60)

file_groups = {
    'player_stats': (FIRST_SEASON, LAST_SEASON),
    'player_bio': (FIRST_SEASON, LAST_SEASON),
    'tracking_stats': (TRACKING_DATA_START, LAST_SEASON),
    'team_schedules': (FIRST_SEASON, LAST_SEASON),
}

for prefix, (start, end) in file_groups.items():
    count = 0
    for season in range(start, end + 1):
        filepath = f'../{RAW_NBA_API_DIR}/{prefix}_{season}.csv'
        if os.path.exists(filepath):
            count += 1
    expected = end - start + 1
    status = "OK" if count == expected else f"MISSING {expected - count}"
    print(f"  {prefix}: {count}/{expected} files [{status}]")

Raw nba_api files available:
  player_stats: 10/10 files [OK]
  player_bio: 10/10 files [OK]
  tracking_stats: 7/7 files [OK]
  team_schedules: 10/10 files [OK]


---
## Section 2: Clean Injury Data (CRITICAL)

The `missed_games_2010_2019.csv` from elap733 has **one row per missed game per player**. We need to:

1. Keep only `Relinquished` rows (player placed on IL, i.e., missed the game)
2. Remove coaches/staff who appear in transaction data
3. Parse calendar dates into NBA seasons (Oct-Jun = one season)
4. Separate genuine injuries from rest/load management
5. Aggregate to **player-season level** — this becomes our target variable

In [4]:
# Examine the missed games data
print("Missed Games Data Structure:")
print(df_missed_games_raw.head(10))
print(f"\nColumns: {list(df_missed_games_raw.columns)}")
print(f"\nDate range: {df_missed_games_raw['Date'].min()} to {df_missed_games_raw['Date'].max()}")

Missed Games Data Structure:
         Date       Team Acquired       Relinquished                                              Notes
0  2010-08-03   Clippers      NaN        Craig Smith  arthroscopic surgery on right knee (out indefi...
1  2010-08-13  Mavericks      NaN  Rodrigue Beaubois  surgery on left foot to repair broken fifth me...
2  2010-08-14   Warriors      NaN          Ekpe Udoh           surgery on left wrist (out indefinitely)
3  2010-09-21    Raptors      NaN       Ed Davis (a)  arthroscopic surgery on right kene to repair t...
4  2010-09-21    Thunder      NaN       Nenad Krstic  surgery on right hand to repair broken finger ...
5  2010-09-24    Bobcats      NaN        Kwame Brown             sprained left ankle (out indefinitely)
6  2010-09-27     Knicks      NaN         Eddy Curry        strained right hamstring (out indefinitely)
7  2010-09-27    Rockets      NaN        Brad Miller                          sprained left ankle (DTD)
8  2010-09-29      Magic      NaN  

NBA seasons span two calendar years (e.g., Oct 2015 – Jun 2016 = "2015-16"). July–September events are assigned to the upcoming season since they typically represent offseason surgeries that affect the next year.

In [5]:
def parse_nba_season(date_str):
    """
    Convert a date string to NBA season string.
    NBA season runs Oct-Jun, so:
    - Oct-Dec 2015 -> '2015-16'
    - Jan-Jun 2016 -> '2015-16'
    - Jul-Sep = offseason, assign to upcoming season
    """
    try:
        date = pd.to_datetime(date_str)
        year = date.year
        month = date.month
        
        if month >= 10:  # Oct-Dec: first year of season
            return f"{year}-{str(year + 1)[-2:]}"
        elif month <= 6:  # Jan-Jun: second year of season
            return f"{year - 1}-{str(year)[-2:]}"
        else:  # Jul-Sep: offseason, assign to upcoming season
            return f"{year}-{str(year + 1)[-2:]}"
    except:
        return None

def get_season_start_year(season_str):
    """Extract the start year from season string like '2015-16' -> 2015"""
    if season_str and '-' in str(season_str):
        return int(season_str.split('-')[0])
    return None

# Test
print("Season parsing tests:")
print(f"  2015-10-15 -> {parse_nba_season('2015-10-15')}")
print(f"  2016-01-15 -> {parse_nba_season('2016-01-15')}")
print(f"  2016-07-15 -> {parse_nba_season('2016-07-15')}")

Season parsing tests:
  2015-10-15 -> 2015-16
  2016-01-15 -> 2015-16
  2016-07-15 -> 2016-17


In [6]:
def extract_injury_type(notes):
    """
    Extract injury type/body part from the Notes column.
    Returns a list of injury keywords found.
    """
    if pd.isna(notes):
        return []
    
    notes_lower = str(notes).lower()
    
    # Body parts to look for
    body_parts = [
        'knee', 'ankle', 'foot', 'achilles', 'hamstring', 'groin',
        'back', 'shoulder', 'wrist', 'hand', 'finger', 'hip',
        'quad', 'calf', 'thigh', 'leg', 'arm', 'elbow',
        'neck', 'head', 'concussion', 'eye', 'nose',
        'toe', 'heel', 'shin', 'rib', 'abdomen', 'abdominal'
    ]
    
    # Injury types
    injury_types = [
        'sprain', 'strain', 'tear', 'fracture', 'broken',
        'surgery', 'sore', 'soreness', 'inflammation',
        'contusion', 'bruise', 'tendinitis', 'tendonitis'
    ]
    
    found = []
    for part in body_parts:
        if part in notes_lower:
            found.append(part)
    
    for inj in injury_types:
        if inj in notes_lower:
            found.append(inj)
    
    return found

def is_rest_day(notes):
    """Check if the entry is rest/load management rather than injury."""
    if pd.isna(notes):
        return False
    notes_lower = str(notes).lower()
    rest_keywords = ['rest', 'load management', 'personal reasons', 'not with team']
    return any(kw in notes_lower for kw in rest_keywords)

# Test
print("Injury extraction tests:")
print(f"  'sprained left ankle (out indefinitely)' -> {extract_injury_type('sprained left ankle (out indefinitely)')}")
print(f"  'torn ACL in right knee' -> {extract_injury_type('torn ACL in right knee')}")
print(f"  'rest (DNP)' -> is_rest: {is_rest_day('rest (DNP)')}")

Injury extraction tests:
  'sprained left ankle (out indefinitely)' -> ['ankle', 'sprain']
  'torn ACL in right knee' -> ['knee']
  'rest (DNP)' -> is_rest: True


In [7]:
def clean_player_name(name):
    """Standardize player name for matching."""
    if pd.isna(name):
        return None
    
    name = str(name).strip()
    
    # Handle name changes (e.g., "Jeff Pendergraph / Jeff Ayres")
    if '/' in name:
        name = name.split('/')[0].strip()
    
    # Remove all parenthesized tokens: disambiguation suffixes like (a), (b),
    # legal-name prefixes like (William), and tags like (T.), (Lamont), (Martin)
    name = re.sub(r'\s*\([^)]*\)', '', name).strip()
    
    # Standardize case
    name = ' '.join(word.capitalize() for word in name.split())
    
    return name

# Test
print("Name cleaning tests:")
print(f"  'Ed Davis (a)' -> '{clean_player_name('Ed Davis (a)')}'")
print(f"  'Jeff Pendergraph / Jeff Ayres' -> '{clean_player_name('Jeff Pendergraph / Jeff Ayres')}'")
print(f"  '(William) Tony Parker' -> '{clean_player_name('(William) Tony Parker')}'")
print(f"  'Marcus Thornton (T.)' -> '{clean_player_name('Marcus Thornton (T.)')}'")
print(f"  'Mike James (Lamont)' -> '{clean_player_name('Mike James (Lamont)')}'")


Name cleaning tests:
  'Ed Davis (a)' -> 'Ed Davis'
  'Jeff Pendergraph / Jeff Ayres' -> 'Jeff Pendergraph'
  '(William) Tony Parker' -> 'Tony Parker'
  'Marcus Thornton (T.)' -> 'Marcus Thornton'
  'Mike James (Lamont)' -> 'Mike James'


Now we apply all cleaning functions to the raw injury data — standardizing names, removing coaches/staff, parsing seasons, and extracting injury types.

In [8]:
# Process missed games data
print("Processing missed games data...")

# Filter to only rows where a player is "Relinquished" (missed a game)
df_missed = df_missed_games_raw[df_missed_games_raw['Relinquished'].notna()].copy()
print(f"Rows with player relinquished: {len(df_missed)}")

# Clean player names
df_missed['player_name'] = df_missed['Relinquished'].apply(clean_player_name)

# Filter out coaches/executives (these appear in transaction data but aren't players)
coaches_removed = df_missed['player_name'].isin(EXCLUDE_FROM_INJURY_DATA).sum()
df_missed = df_missed[~df_missed['player_name'].isin(EXCLUDE_FROM_INJURY_DATA)]
print(f"Removed {coaches_removed} entries for coaches/staff")

# Parse season
df_missed['season'] = df_missed['Date'].apply(parse_nba_season)
df_missed['season_start_year'] = df_missed['season'].apply(get_season_start_year)

# Extract injury info
df_missed['injury_types'] = df_missed['Notes'].apply(extract_injury_type)
df_missed['is_rest'] = df_missed['Notes'].apply(is_rest_day)

print(f"\nSample processed data:")
df_missed[['Date', 'player_name', 'season', 'Notes', 'injury_types', 'is_rest']].head(10)

Processing missed games data...
Rows with player relinquished: 9328
Removed 21 entries for coaches/staff



Sample processed data:


,Date,player_name,season,Notes,injury_types,is_rest
0,2010-08-03,Craig Smith,2010-11,arthroscopic surgery on right knee (out indefi...,"[knee, surgery]",False
1,2010-08-13,Rodrigue Beaubois,2010-11,surgery on left foot to repair broken fifth me...,"[foot, broken, surgery]",False
2,2010-08-14,Ekpe Udoh,2010-11,surgery on left wrist (out indefinitely),"[wrist, surgery]",False
3,2010-09-21,Ed Davis,2010-11,arthroscopic surgery on right kene to repair t...,[surgery],False
4,2010-09-21,Nenad Krstic,2010-11,surgery on right hand to repair broken finger ...,"[hand, finger, broken, surgery]",False
5,2010-09-24,Kwame Brown,2010-11,sprained left ankle (out indefinitely),"[ankle, sprain]",False
6,2010-09-27,Eddy Curry,2010-11,strained right hamstring (out indefinitely),"[hamstring, strain]",False
7,2010-09-27,Brad Miller,2010-11,sprained left ankle (DTD),"[ankle, sprain]",False
8,2010-09-29,Jason Williams,2010-11,arthroscopic surgery on left knee (out indefin...,"[knee, surgery]",False
9,2010-10-03,Carlos Boozer,2010-11,fractured bone in right pinky finger (out inde...,"[finger, fracture]",False


We exclude rest days because load management is not injury-related and would inflate the target variable. We're predicting injury risk, not coaching decisions.

In [9]:
# Check rest vs injury breakdown
print("Rest vs Injury breakdown:")
print(df_missed['is_rest'].value_counts())
print(f"\nRest percentage: {df_missed['is_rest'].mean()*100:.1f}%")

Rest vs Injury breakdown:
is_rest
False    8356
True      951
Name: count, dtype: int64

Rest percentage: 10.2%


In [10]:
# Exclude pure rest days, keep injuries only
df_injuries_only = df_missed[~df_missed['is_rest']].copy()
print(f"After excluding rest days: {len(df_injuries_only)} injury events")

After excluding rest days: 8356 injury events


In [11]:
# Filter to our season range (2010-2019)
df_injuries_only = df_injuries_only[
    (df_injuries_only['season_start_year'] >= FIRST_SEASON) & 
    (df_injuries_only['season_start_year'] <= LAST_SEASON)
].copy()

print(f"After filtering to {FIRST_SEASON}-{LAST_SEASON}: {len(df_injuries_only)} injury events")
print(f"\nSeasons covered:")
print(df_injuries_only['season'].value_counts().sort_index())

After filtering to 2010-2019: 8356 injury events

Seasons covered:
season
2010-11     823
2011-12    1310
2012-13    1227
2013-14    1742
2014-15     615
2015-16     721
2016-17     731
2017-18     622
2018-19     564
2019-20       1
Name: count, dtype: int64


Now we aggregate from one-row-per-missed-game to **one-row-per-player-per-season**. Each row's `games_missed` count becomes the basis for our target variable.

In [12]:
# Aggregate to player-season level
print("Aggregating to player-season level...")

def aggregate_injuries(group):
    """Aggregate injury data for a player-season."""
    all_injuries = []
    for inj_list in group['injury_types']:
        all_injuries.extend(inj_list)
    
    unique_injuries = list(set(all_injuries))
    
    return pd.Series({
        'games_missed': len(group),
        'injury_events': group['Notes'].nunique(),
        'injury_types': unique_injuries,
        'teams': list(group['Team'].unique())
    })

df_injury_by_player_season = df_injuries_only.groupby(
    ['player_name', 'season', 'season_start_year'],
    as_index=False
).apply(aggregate_injuries, include_groups=False).reset_index(drop=True)

print(f"\nPlayer-season injury records: {len(df_injury_by_player_season)}")
df_injury_by_player_season.head(10)

Aggregating to player-season level...



Player-season injury records: 2622


,player_name,season,season_start_year,games_missed,injury_events,injury_types,teams
0,A.j. Price,2011-12,2011,2,1,[],[Pacers]
1,A.j. Price,2012-13,2012,15,4,"[sore, hand, knee, fracture, groin]",[Wizards]
2,Aaron Brooks,2012-13,2012,1,1,"[sore, ankle]",[Kings]
3,Aaron Brooks,2013-14,2013,1,1,"[tendinitis, knee]",[Rockets]
4,Aaron Brooks,2015-16,2015,2,1,"[strain, hamstring]",[Bulls]
5,Aaron Brooks,2016-17,2016,2,1,"[sore, knee]",[Pacers]
6,Aaron Gordon,2014-15,2014,1,1,"[fracture, foot]",[Magic]
7,Aaron Gordon,2015-16,2015,2,2,"[concussion, ankle]",[Magic]
8,Aaron Gordon,2016-17,2016,1,1,"[bruise, foot]",[Magic]
9,Aaron Gordon,2017-18,2017,5,4,"[strain, sore, calf, concussion, hip]",[Magic]


In [13]:
# Summary statistics
print("Injury Summary Statistics:")
print(f"\nGames missed per player-season:")
print(df_injury_by_player_season['games_missed'].describe())

print(f"\nTop 10 most injury-prone player-seasons:")
df_injury_by_player_season.nlargest(10, 'games_missed')[['player_name', 'season', 'games_missed', 'injury_types']]

Injury Summary Statistics:

Games missed per player-season:
count    2622.000000
mean        3.186880
std         3.750642
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        40.000000
Name: games_missed, dtype: float64

Top 10 most injury-prone player-seasons:


,player_name,season,games_missed,injury_types
1463,Kevin Love,2012-13,40,"[eye, hand, surgery, knee, finger, fracture]"
942,Greg Smith,2013-14,39,"[sprain, sore, knee, surgery, hip]"
1142,Jason Smith,2013-14,39,"[bruise, knee]"
1873,Nate Robinson,2013-14,37,"[surgery, knee]"
1314,Jordan Farmar,2013-14,31,"[groin, strain, sore, hamstring]"
772,Ekpe Udoh,2013-14,28,"[sprain, ankle, surgery, knee]"
2305,Steve Blake,2013-14,27,[elbow]
2540,Vitor Faverani,2013-14,27,[knee]
2408,Tony Allen,2013-14,26,"[hand, wrist, hip, sprain]"
791,Emeka Okafor,2011-12,25,"[sore, knee]"


In [14]:
# Sanity checks
assert df_injury_by_player_season['games_missed'].min() >= 0, "Negative games_missed found!"
assert df_injury_by_player_season['games_missed'].max() <= 82, "games_missed exceeds season length!"
assert len(df_injury_by_player_season) > 2000, "Too few injury records — data may be incomplete"
print("All injury data assertions passed.")

All injury data assertions passed.


In [15]:
# Save injury history
output_path = f'../{PROCESSED_DIR}/injury_history_by_player_season.csv'
df_injury_by_player_season.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_injury_by_player_season.shape}")

Saved: ../data/processed/injury_history_by_player_season.csv
Shape: (2622, 7)


### Section 2 Summary

**Decisions made:**
- **Excluded rest days** (10.2% of entries) — load management is not injury, and including it would inflate the target
- **Excluded coaches/staff** who appear in transaction data (not players)
- **Counted each row as one game missed** — the elap733 data has one row per DNP event
- **Aggregated to player-season level** with games missed, injury count, and injury types

**Output:** `injury_history_by_player_season.csv` (2,623 rows)

---
## Section 3: Clean Player Stats

Combine per-season player stats files from nba_api into a single dataframe. We check for mid-season trade duplicates (a player traded mid-season could appear twice for the same season) and standardize column names.

In [16]:
# Load and combine all player stats files
print("Combining player stats files...")

player_stats_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'../{RAW_NBA_API_DIR}/player_stats_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        player_stats_dfs.append(df)

df_player_stats = pd.concat(player_stats_dfs, ignore_index=True)
print(f"Combined {len(player_stats_dfs)} files -> shape: {df_player_stats.shape}")
print(f"Columns: {list(df_player_stats.columns)}")

Combining player stats files...
Combined 10 files -> shape: (4934, 69)
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT', 'SEASON', 'SEASON_START_YEAR']


In [17]:
# Check for duplicates — players traded mid-season could appear twice
dupes = df_player_stats.duplicated(subset=['PLAYER_ID', 'SEASON'], keep=False)
print(f"Duplicate player-season records: {dupes.sum()}")

if dupes.sum() > 0:
    print("\nSample duplicates:")
    print(df_player_stats[dupes][['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 'SEASON']].head(10))
    
    # Keep the row with most games played for each player-season
    print("\nKeeping row with most games played for each player-season...")
    df_player_stats = df_player_stats.sort_values('GP', ascending=False).drop_duplicates(
        subset=['PLAYER_ID', 'SEASON'], keep='first'
    )
    print(f"Shape after deduplication: {df_player_stats.shape}")

Duplicate player-season records: 0


In [18]:
# Standardize column names (lowercase)
df_player_stats.columns = df_player_stats.columns.str.lower().str.replace(' ', '_')

# Clean player names for matching
df_player_stats['player_name_clean'] = df_player_stats['player_name'].apply(clean_player_name)

print(f"Final columns ({len(df_player_stats.columns)}): {list(df_player_stats.columns[:15])} ...")
df_player_stats.head(3)

Final columns (70): ['player_id', 'player_name', 'nickname', 'team_id', 'team_abbreviation', 'age', 'gp', 'w', 'l', 'w_pct', 'min', 'fgm', 'fga', 'fg_pct', 'fg3m'] ...


,player_id,player_name,nickname,team_id,team_abbreviation,age,gp,w,l,w_pct,min,fgm,fga,fg_pct,fg3m,fg3a,fg3_pct,ftm,fta,ft_pct,oreb,dreb,reb,ast,tov,...,fg3a_rank,fg3_pct_rank,ftm_rank,fta_rank,ft_pct_rank,oreb_rank,dreb_rank,reb_rank,ast_rank,tov_rank,stl_rank,blk_rank,blka_rank,pf_rank,pfd_rank,pts_rank,plus_minus_rank,nba_fantasy_pts_rank,dd2_rank,td3_rank,wnba_fantasy_pts_rank,team_count,season,season_start_year,player_name_clean
0,201985,AJ Price,AJ,1610612754,IND,24.0,50,22,28,0.440,15.9,2.3,6.4,0.356,0.8,3.0,0.275,1.1,1.6,0.667,0.3,1.1,1.4,2.2,1.1,...,97,241,234,212,316,348,369,375,121,203,208,421,308,351,198,240,161,274,224,27,264,1,2010-11,2010,Aj Price
1,201166,Aaron Brooks,Aaron,1610612756,PHX,26.0,59,26,33,0.441,21.8,3.7,9.9,0.375,1.2,4.0,0.297,2.1,2.4,0.886,0.3,1.0,1.3,3.9,1.7,...,55,227,104,138,30,344,388,385,44,99,209,392,62,205,131,132,377,179,154,27,164,2,2010-11,2010,Aaron Brooks
2,201189,Aaron Gray,Aaron,1610612740,NOH,26.0,41,21,20,0.512,12.9,1.4,2.4,0.566,0.0,0.0,0.000,0.4,0.8,0.500,1.4,2.7,4.2,0.4,0.8,...,383,309,367,333,398,98,169,133,376,290,368,223,270,121,348,362,145,326,154,27,332,1,2010-11,2010,Aaron Gray


In [19]:
# Check for missing values in key columns
key_cols = ['player_id', 'player_name', 'season', 'gp', 'min', 'pts']
print("Missing values in key columns:")
print(df_player_stats[key_cols].isnull().sum())

Missing values in key columns:
player_id      0
player_name    0
season         0
gp             0
min            0
pts            0
dtype: int64


In [20]:
# Assertions
assert df_player_stats.duplicated(subset=['player_id', 'season']).sum() == 0, \
    "Duplicate player-season records remain after dedup!"
print("No duplicate player-season records. Good.")

# Save combined player stats
output_path = f'../{PROCESSED_DIR}/player_stats_combined.csv'
df_player_stats.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_player_stats.shape}")

No duplicate player-season records. Good.
Saved: ../data/processed/player_stats_combined.csv
Shape: (4934, 70)


---
## Section 4: Clean Player Bio

Player bio includes height, weight, college, country, draft info, and advanced rate stats (USG%, TS%, etc.). We lowercase column names to match player stats and check for duplicates.

In [21]:
# Load and combine all player bio files
print("Combining player bio files...")

player_bio_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'../{RAW_NBA_API_DIR}/player_bio_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        player_bio_dfs.append(df)

df_player_bio = pd.concat(player_bio_dfs, ignore_index=True)
print(f"Combined {len(player_bio_dfs)} files -> shape: {df_player_bio.shape}")
print(f"Columns (raw): {list(df_player_bio.columns)}")

Combining player bio files...
Combined 10 files -> shape: (4934, 25)
Columns (raw): ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GP', 'PTS', 'REB', 'AST', 'NET_RATING', 'OREB_PCT', 'DREB_PCT', 'USG_PCT', 'TS_PCT', 'AST_PCT', 'SEASON', 'SEASON_START_YEAR']


In [22]:
# Standardize column names to lowercase (matching player_stats_combined)
df_player_bio.columns = df_player_bio.columns.str.lower().str.replace(' ', '_')

# Clean player names for matching
df_player_bio['player_name_clean'] = df_player_bio['player_name'].apply(clean_player_name)

print(f"Columns (standardized): {list(df_player_bio.columns)}")

Columns (standardized): ['player_id', 'player_name', 'team_id', 'team_abbreviation', 'age', 'player_height', 'player_height_inches', 'player_weight', 'college', 'country', 'draft_year', 'draft_round', 'draft_number', 'gp', 'pts', 'reb', 'ast', 'net_rating', 'oreb_pct', 'dreb_pct', 'usg_pct', 'ts_pct', 'ast_pct', 'season', 'season_start_year', 'player_name_clean']


In [23]:
# Check for duplicates
dupes = df_player_bio.duplicated(subset=['player_id', 'season'], keep=False)
print(f"Duplicate player-season records in bio: {dupes.sum()}")

if dupes.sum() > 0:
    df_player_bio = df_player_bio.drop_duplicates(subset=['player_id', 'season'], keep='first')
    print(f"Shape after deduplication: {df_player_bio.shape}")

Duplicate player-season records in bio: 0


In [24]:
# Save combined player bio
output_path = f'../{PROCESSED_DIR}/player_bio_combined.csv'
df_player_bio.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_player_bio.shape}")

Saved: ../data/processed/player_bio_combined.csv
Shape: (4934, 26)


---
## Section 5: Clean Tracking Data (2013+ only)

NBA.com player tracking (distance, speed) started in the 2013-14 season, so this data only covers 7 of our 10 seasons. Features derived from tracking will have ~28% missing values for the 2010-2012 seasons. Downstream notebooks must decide how to handle this gap (imputation, exclusion, or separate modeling).

In [25]:
# Load and combine tracking stats (2013+ only)
print("Combining tracking stats files...")

tracking_stats_dfs = []
for season in range(TRACKING_DATA_START, LAST_SEASON + 1):
    filepath = f'../{RAW_NBA_API_DIR}/tracking_stats_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        tracking_stats_dfs.append(df)

if tracking_stats_dfs:
    df_tracking = pd.concat(tracking_stats_dfs, ignore_index=True)
    print(f"Combined {len(tracking_stats_dfs)} files -> shape: {df_tracking.shape}")
    print(f"Columns: {list(df_tracking.columns)}")
else:
    print("No tracking stats files found!")
    df_tracking = pd.DataFrame()

Combining tracking stats files...


Combined 7 files -> shape: (3535, 18)
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W', 'L', 'MIN', 'MIN1', 'DIST_FEET', 'DIST_MILES', 'DIST_MILES_OFF', 'DIST_MILES_DEF', 'AVG_SPEED', 'AVG_SPEED_OFF', 'AVG_SPEED_DEF', 'SEASON', 'SEASON_START_YEAR']


In [26]:
if not df_tracking.empty:
    # Standardize column names
    df_tracking.columns = df_tracking.columns.str.lower().str.replace(' ', '_')
    
    # Clean player names
    df_tracking['player_name_clean'] = df_tracking['player_name'].apply(clean_player_name)
    
    # Handle duplicates
    dupes = df_tracking.duplicated(subset=['player_id', 'season'], keep=False)
    print(f"Duplicate player-season records: {dupes.sum()}")
    
    if dupes.sum() > 0:
        df_tracking = df_tracking.drop_duplicates(subset=['player_id', 'season'], keep='first')
    
    print(f"\nSeasons in tracking data:")
    print(df_tracking['season'].value_counts().sort_index())

Duplicate player-season records: 0

Seasons in tracking data:
season
2013-14    482
2014-15    492
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
Name: count, dtype: int64


In [27]:
# Save combined tracking data
if not df_tracking.empty:
    output_path = f'../{PROCESSED_DIR}/tracking_stats_combined.csv'
    df_tracking.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    print(f"Shape: {df_tracking.shape}")

Saved: ../data/processed/tracking_stats_combined.csv
Shape: (3535, 19)


---
## Section 6: Calculate Back-to-Back Games

Back-to-back games (playing on consecutive days) increase fatigue and injury risk. We flag every B2B game in the schedule, then aggregate B2B counts per team-season for use as a feature.

In [28]:
# Load and combine team schedule files
print("Combining team schedule files...")

team_schedules_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'../{RAW_NBA_API_DIR}/team_schedules_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        team_schedules_dfs.append(df)

df_schedules = pd.concat(team_schedules_dfs, ignore_index=True)
print(f"Combined {len(team_schedules_dfs)} files -> shape: {df_schedules.shape}")
print(f"Columns: {list(df_schedules.columns)}")

Combining team schedule files...


Combined 10 files -> shape: (23778, 30)
Columns: ['Team_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'TEAM_ABBREVIATION', 'SEASON', 'SEASON_START_YEAR']


In [29]:
# Standardize column names
df_schedules.columns = df_schedules.columns.str.lower().str.replace(' ', '_')

# Parse game date with explicit format handling
df_schedules['game_date'] = pd.to_datetime(df_schedules['game_date'])

# Sort by team and date
df_schedules = df_schedules.sort_values(['team_id', 'game_date'])

# Calculate days since last game for each team
df_schedules['days_rest'] = df_schedules.groupby('team_id')['game_date'].diff().dt.days

# Flag back-to-back games (played day after previous game)
df_schedules['is_back_to_back'] = (df_schedules['days_rest'] == 1).astype(int)

print(f"Back-to-back game frequency:")
print(df_schedules['is_back_to_back'].value_counts())
print(f"\nB2B percentage: {df_schedules['is_back_to_back'].mean()*100:.1f}%")

Back-to-back game frequency:
is_back_to_back
0    18677
1     5101
Name: count, dtype: int64

B2B percentage: 21.5%


/var/folders/tt/q6yxx0qn7dq_jbphcnvcmtfr0000gn/T/ipykernel_10585/343097084.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_schedules['game_date'] = pd.to_datetime(df_schedules['game_date'])


In [30]:
# Aggregate back-to-backs by team-season
df_b2b_summary = df_schedules.groupby(['team_id', 'team_abbreviation', 'season']).agg({
    'is_back_to_back': 'sum',
    'game_date': 'count'
}).reset_index()

df_b2b_summary.columns = ['team_id', 'team_abbreviation', 'season', 'b2b_games', 'total_games']
df_b2b_summary['b2b_percentage'] = df_b2b_summary['b2b_games'] / df_b2b_summary['total_games'] * 100

print(f"B2B summary shape: {df_b2b_summary.shape}")
print(f"\nAbout 21.5% of all NBA games are back-to-backs.")
df_b2b_summary.head(10)

B2B summary shape: (300, 6)

About 21.5% of all NBA games are back-to-backs.


,team_id,team_abbreviation,season,b2b_games,total_games,b2b_percentage
0,1610612737,ATL,2010-11,23,82,28.048780
1,1610612737,ATL,2011-12,19,66,28.787879
2,1610612737,ATL,2012-13,22,82,26.829268
3,1610612737,ATL,2013-14,21,82,25.609756
4,1610612737,ATL,2014-15,21,82,25.609756
5,1610612737,ATL,2015-16,19,82,23.170732
6,1610612737,ATL,2016-17,18,82,21.951220
7,1610612737,ATL,2017-18,15,82,18.292683
8,1610612737,ATL,2018-19,12,82,14.634146
9,1610612737,ATL,2019-20,12,67,17.910448


In [31]:
# Save team B2B summary
output_path = f'../{PROCESSED_DIR}/team_back_to_backs.csv'
df_b2b_summary.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_b2b_summary.shape}")

# Save full schedule with B2B flags
output_path = f'../{PROCESSED_DIR}/team_schedules_with_b2b.csv'
df_schedules.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_schedules.shape}")

Saved: ../data/processed/team_back_to_backs.csv
Shape: (300, 6)


Saved: ../data/processed/team_schedules_with_b2b.csv
Shape: (23778, 32)


---
## Section 7: Create Player ID Mapping (CRITICAL)

**The core challenge:** elap733 injury data identifies players by name only, while nba_api uses numeric `PLAYER_ID`. We must bridge these two systems to merge injury history with player stats.

**Strategy (3-tier):**
1. **Manual mappings** (39 entries) — handles special characters (Jokić), nicknames (Bam Adebayo), and suffix mismatches (Jr./III)
2. **Exact match** on cleaned names
3. **Fuzzy match** (SequenceMatcher, threshold ≥ 0.85) as a fallback for remaining discrepancies

In [32]:
# Get unique players from each source
print("Unique players in each dataset:")

# From injury data — apply manual mappings
injury_players_raw = set(df_injury_by_player_season['player_name'].unique())
print(f"  Injury data (raw): {len(injury_players_raw)} unique players")

# Apply manual name mappings to injury player names
injury_players_mapped = {}
for player in injury_players_raw:
    mapped_name = MANUAL_NAME_MAPPINGS.get(player, player)
    injury_players_mapped[player] = mapped_name
    if player != mapped_name:
        print(f"    Mapped: '{player}' -> '{mapped_name}'")

injury_players = set(injury_players_mapped.values())
print(f"  Injury data (after mapping): {len(injury_players)} unique players")

# From nba_api stats
nba_players = df_player_stats[['player_id', 'player_name', 'player_name_clean']].drop_duplicates()
print(f"  NBA API stats: {len(nba_players)} unique players")

Unique players in each dataset:
  Injury data (raw): 849 unique players
    Mapped: 'Jonas Valanciunas' -> 'Jonas Valančiūnas'
    Mapped: 'Mike Conley Jr.' -> 'Mike Conley'
    Mapped: 'Sviatoslav Mykhailiuk' -> 'Svi Mykhailiuk'
    Mapped: 'Goran Dragic' -> 'Goran Dragić'
    Mapped: 'Bogdan Bogdanovic' -> 'Bogdan Bogdanović'
    Mapped: 'Predrag Stojakovic' -> 'Peja Stojakovic'
    Mapped: 'Mohamed Bamba' -> 'Mo Bamba'
    Mapped: 'Jusuf Nurkic' -> 'Jusuf Nurkić'
    Mapped: 'Luka Doncic' -> 'Luka Dončić'
    Mapped: 'Maximilian Kleber' -> 'Maxi Kleber'
    Mapped: 'James Ennis' -> 'James Ennis III'
    Mapped: 'Edrice Adebayo' -> 'Bam Adebayo'
    Mapped: 'Dario Saric' -> 'Dario Šarić'
    Mapped: 'Maurice Williams' -> 'Mo Williams'
    Mapped: 'Mike Dunleavy Jr.' -> 'Mike Dunleavy'
    Mapped: 'Luigi Datome' -> 'Gigi Datome'
    Mapped: 'Raulzinho Neto' -> 'Raul Neto'
    Mapped: 'Hidayet Turkoglu' -> 'Hedo Turkoglu'
    Mapped: 'Tony Wroten Jr.' -> 'Tony Wroten'
    Mapped: 'Niko

Exact name matching won't catch all players due to special characters (Nikola Jokić), nicknames (Bam Adebayo), and suffixes (Jr./III). We use fuzzy matching as a fallback with a conservative threshold of 0.85 to avoid false positives.

In [33]:
def fuzzy_match_score(name1, name2):
    """Calculate similarity score between two names."""
    if not name1 or not name2:
        return 0
    return SequenceMatcher(None, name1.lower(), name2.lower()).ratio()

def find_best_match(name, candidates, threshold=0.85):
    """
    Find the best matching name from candidates.
    Returns (matched_name, score, player_id) or (None, 0, None) if no match.
    """
    best_match = None
    best_score = 0
    best_id = None
    
    for _, row in candidates.iterrows():
        candidate_name = row['player_name_clean']
        score = fuzzy_match_score(name, candidate_name)
        
        if score > best_score:
            best_score = score
            best_match = candidate_name
            best_id = row['player_id']
    
    if best_score >= threshold:
        return best_match, best_score, best_id
    return None, best_score, None

# Test fuzzy matching
print("Fuzzy matching tests:")
print(f"  'LeBron James' vs 'Lebron James': {fuzzy_match_score('LeBron James', 'Lebron James'):.3f}")
print(f"  'Stephen Curry' vs 'Steph Curry': {fuzzy_match_score('Stephen Curry', 'Steph Curry'):.3f}")

Fuzzy matching tests:
  'LeBron James' vs 'Lebron James': 1.000
  'Stephen Curry' vs 'Steph Curry': 0.917


In [34]:
# Create player ID mapping
print("Creating player ID mapping...")
print("This may take a few minutes for fuzzy matching...")

mapping_results = []
unmatched = []

# Process each original injury player name
for i, (original_name, mapped_name) in enumerate(injury_players_mapped.items()):
    if i % 100 == 0:
        print(f"  Processing {i}/{len(injury_players_mapped)}...")
    
    # First try exact match with mapped name
    exact_match = nba_players[nba_players['player_name_clean'] == mapped_name]
    
    # Also try matching against player_name directly (for special characters)
    if len(exact_match) == 0:
        exact_match = nba_players[nba_players['player_name'] == mapped_name]
    
    if len(exact_match) > 0:
        # Exact match found
        match_type = 'manual' if original_name != mapped_name else 'exact'
        mapping_results.append({
            'injury_player_name': original_name,
            'nba_player_name': exact_match.iloc[0]['player_name'],
            'nba_player_id': exact_match.iloc[0]['player_id'],
            'match_type': match_type,
            'match_score': 1.0
        })
    else:
        # Try fuzzy match
        matched_name, score, player_id = find_best_match(mapped_name, nba_players)
        
        if matched_name:
            mapping_results.append({
                'injury_player_name': original_name,
                'nba_player_name': matched_name,
                'nba_player_id': player_id,
                'match_type': 'fuzzy',
                'match_score': score
            })
        else:
            unmatched.append({
                'injury_player_name': original_name,
                'mapped_name': mapped_name,
                'best_score': score
            })

print(f"\nMatching complete!")
print(f"  Matched: {len(mapping_results)}")
print(f"  Unmatched: {len(unmatched)}")

Creating player ID mapping...
This may take a few minutes for fuzzy matching...
  Processing 0/849...
  Processing 100/849...


  Processing 200/849...


  Processing 300/849...


  Processing 400/849...


  Processing 500/849...


  Processing 600/849...


  Processing 700/849...


  Processing 800/849...

Matching complete!
  Matched: 843
  Unmatched: 6


In [35]:
# Create mapping dataframe
df_player_mapping = pd.DataFrame(mapping_results)

print("Match type breakdown:")
print(df_player_mapping['match_type'].value_counts())

print(f"\nFuzzy matches (review these for false positives):")
fuzzy_matches = df_player_mapping[df_player_mapping['match_type'] == 'fuzzy'].sort_values('match_score')
print(fuzzy_matches[['injury_player_name', 'nba_player_name', 'match_score']].to_string())

Match type breakdown:
match_type
exact     780
manual     34
fuzzy      29
Name: count, dtype: int64

Fuzzy matches (review these for false positives):
     injury_player_name     nba_player_name  match_score
124        Danuel House    Danuel House Jr.     0.857143
137        Jimmy Butler    Jimmy Butler Iii     0.857143
337    Roy Devyn Marble        Devyn Marble     0.857143
495       Marcus Morris   Marcus Morris Sr.     0.866667
174          Kevin Knox       Kevin Knox Ii     0.869565
620       Wesley Iwundu          Wes Iwundu     0.869565
568      Reggie Bullock  Reggie Bullock Jr.     0.875000
651          J.r. Smith            Jr Smith     0.888889
281          D.j. White            Dj White     0.888889
648          A.j. Price            Aj Price     0.888889
743          C.j. Miles            Cj Miles     0.888889
147    Emanuel Ginobili       Manu Ginobili     0.896552
489          Dante Exum          Danté Exum     0.900000
631         R.j. Hunter           Rj Hunter     0.

In [36]:
# Save player mapping
output_path = f'../{PROCESSED_DIR}/player_id_mapping.csv'
df_player_mapping.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_player_mapping.shape}")

print(f"\nMatch type breakdown:")
print(df_player_mapping['match_type'].value_counts())

# Save unmatched for manual review
df_unmatched = pd.DataFrame(unmatched)
if len(df_unmatched) > 0:
    output_path = f'../{PROCESSED_DIR}/unmatched_players.csv'
    df_unmatched.to_csv(output_path, index=False)
    print(f"\nSaved unmatched players: {output_path}")
    print(df_unmatched)
else:
    # Remove old unmatched file if all players are now matched
    unmatched_path = f'../{PROCESSED_DIR}/unmatched_players.csv'
    if os.path.exists(unmatched_path):
        os.remove(unmatched_path)
        print(f"\nRemoved old unmatched_players.csv (all players now matched)")

Saved: ../data/processed/player_id_mapping.csv
Shape: (843, 5)

Match type breakdown:
match_type
exact     780
manual     34
fuzzy      29
Name: count, dtype: int64

Saved unmatched players: ../data/processed/unmatched_players.csv
  injury_player_name        mapped_name  best_score
0     Walter Tavares     Walter Tavares    0.720000
1    Yakuba Ouattara    Yakuba Ouattara    0.571429
2      Terrico White      Terrico White    0.846154
3  Sheldon Mcclellan  Sheldon Mcclellan    0.714286
4   Jeff Pendergraph   Jeff Pendergraph    0.538462
5     Trevon Bluiett     Trevon Bluiett    0.615385


In [37]:
# Assertions on player mapping quality
total_injury_players = len(injury_players_mapped)
matched_players = len(df_player_mapping)
match_rate = matched_players / total_injury_players * 100

print(f"Player Mapping Coverage:")
print(f"  Players in injury data: {total_injury_players}")
print(f"  Players matched to NBA ID: {matched_players}")
print(f"  Match rate: {match_rate:.1f}%")

assert df_player_mapping.shape[0] >= 840, \
    f"Player match rate dropped below threshold! Only {matched_players} matched."
print("\nPlayer mapping assertion passed.")

Player Mapping Coverage:
  Players in injury data: 849
  Players matched to NBA ID: 843
  Match rate: 99.3%

Player mapping assertion passed.


---
## Section 8: Data Quality Summary

Final verification of all output files — shapes, file sizes, and null counts for key columns.

In [38]:
print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

# Collect all output dataframes and their key columns for null checking
output_info = [
    ('injury_history_by_player_season.csv', df_injury_by_player_season, ['player_name', 'season', 'games_missed']),
    ('player_stats_combined.csv', df_player_stats, ['player_id', 'player_name', 'season', 'gp', 'min']),
    ('player_bio_combined.csv', df_player_bio, ['player_id', 'player_name', 'season', 'player_height_inches', 'player_weight']),
    ('tracking_stats_combined.csv', df_tracking, ['player_id', 'season', 'dist_miles', 'avg_speed']),
    ('team_back_to_backs.csv', df_b2b_summary, ['team_id', 'season', 'b2b_games']),
    ('team_schedules_with_b2b.csv', df_schedules, ['team_id', 'game_date', 'is_back_to_back']),
    ('player_id_mapping.csv', df_player_mapping, ['injury_player_name', 'nba_player_id', 'match_type']),
]

print(f"\n{'File':<45} {'Rows':>6} {'Cols':>5} {'Size':>10}  Key Column Nulls")
print("-" * 110)

for filename, df, key_cols in output_info:
    filepath = f'../{PROCESSED_DIR}/{filename}'
    size_kb = os.path.getsize(filepath) / 1024
    
    # Check nulls in key columns
    null_info = []
    for col in key_cols:
        if col in df.columns:
            n = df[col].isnull().sum()
            if n > 0:
                null_info.append(f"{col}={n}")
    null_str = ', '.join(null_info) if null_info else 'none'
    
    print(f"  {filename:<43} {df.shape[0]:>6} {df.shape[1]:>5} {size_kb:>8.1f}KB  {null_str}")

print(f"\n{'Total files:':<45} {len(output_info)}")

DATA QUALITY SUMMARY

File                                            Rows  Cols       Size  Key Column Nulls
--------------------------------------------------------------------------------------------------------------
  injury_history_by_player_season.csv           2622     7    176.7KB  none
  player_stats_combined.csv                     4934    70   1500.1KB  none
  player_bio_combined.csv                       4934    26    768.2KB  player_height_inches=5, player_weight=5
  tracking_stats_combined.csv                   3535    19    398.3KB  dist_miles=1, avg_speed=1
  team_back_to_backs.csv                         300     6     14.0KB  none
  team_schedules_with_b2b.csv                  23778    32   3298.1KB  none
  player_id_mapping.csv                          843     5     37.6KB  none

Total files:                                  7


In [39]:
# Per-season data quality check
print("\nPer-Season Quality Check:")
print("=" * 60)

players_per_season = df_player_stats.groupby('season').size()
print(f"\nPlayers per season (stats):")
print(players_per_season)

injuries_per_season = df_injury_by_player_season.groupby('season')['games_missed'].sum()
print(f"\nTotal games missed per season (from injury data):")
print(injuries_per_season)

print(f"\n*** NOTE: 2019-20 has only {injuries_per_season.get('2019-20', 0)} game(s) missed ***")
print("The elap733 injury data ends July 2019, so the 2019-20 season has")
print("essentially no injury coverage. This will need handling in feature engineering.")


Per-Season Quality Check:

Players per season (stats):
season
2010-11    452
2011-12    478
2012-13    469
2013-14    482
2014-15    492
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
dtype: int64

Total games missed per season (from injury data):
season
2010-11     823
2011-12    1310
2012-13    1227
2013-14    1742
2014-15     615
2015-16     721
2016-17     731
2017-18     622
2018-19     564
2019-20       1
Name: games_missed, dtype: int64

*** NOTE: 2019-20 has only 1 game(s) missed ***
The elap733 injury data ends July 2019, so the 2019-20 season has
essentially no injury coverage. This will need handling in feature engineering.


---
## Summary

### Output Files (`data/processed/`)

| File | Rows | Description |
|------|------|-------------|
| `injury_history_by_player_season.csv` | ~2,600 | **TARGET VARIABLE** — Games missed per player per season |
| `player_stats_combined.csv` | ~4,900 | Combined per-game player stats (2010-2019) |
| `player_bio_combined.csv` | ~4,900 | Player demographics (height, weight, draft, USG%, TS%) |
| `tracking_stats_combined.csv` | ~3,500 | Speed/distance tracking (2013-2019 only) |
| `team_back_to_backs.csv` | 300 | B2B game counts per team-season |
| `team_schedules_with_b2b.csv` | ~23,800 | Full game schedule with B2B flags |
| `player_id_mapping.csv` | ~840 | Name-to-ID mapping (99%+ match rate) |

### Key Decisions

1. **Excluded rest days** (10.2%) — load management is not injury
2. **Excluded coaches/staff** from injury data
3. **Player matching** achieved ~99% via exact + manual + fuzzy matching
4. **Tracking data** only covers 2013+, creating a gap for 2010-2012

---
### Handoff to Notebook 03

**Files produced:**
- `injury_history_by_player_season.csv` — target variable source
- `player_stats_combined.csv` — player performance features
- `player_bio_combined.csv` — player demographic features
- `tracking_stats_combined.csv` — movement/speed features
- `team_back_to_backs.csv` — schedule fatigue features
- `team_schedules_with_b2b.csv` — game-level B2B flags
- `player_id_mapping.csv` — bridge table for merging

**Expected inputs for NB03 (EDA):**
- `injury_history_by_player_season.csv`
- `player_stats_combined.csv`
- `player_bio_combined.csv`
- `player_id_mapping.csv`

**Known limitations:**
- **Tracking data** only available 2013-14 onward (7 of 10 seasons) — ~28% missing
- **Unmatched players** dropped (~7 minor players with few games)
- **2019-20 season** has near-zero injury coverage because the elap733 data ends July 2019
- **Offseason injuries** (Jul-Sep) are assigned to the upcoming season
- **Fuzzy matches** reviewed manually — all remaining matches appear valid after excluding known false positives